# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [4]:
con.sql(f"""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS n_position_placeholder,
        SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS n_zero_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────────────────┬────────────────────┐
│ n_total │ n_position_placeholder │ n_zero_impressions │
│  int64  │         int128         │       int128       │
├─────────┼────────────────────────┼────────────────────┤
│ 3611061 │                 163189 │                  0 │
└─────────┴────────────────────────┴────────────────────┘

In [5]:
con.sql(f"""
    SELECT
        approx_quantile(gsc_impressions, [0.5, 0.75, 0.9, 0.95, 0.99]) AS impressions_pctiles,
        approx_quantile(
            gsc_clicks / NULLIF(gsc_impressions, 0),
            [0.5, 0.75, 0.9, 0.95, 0.99]
        ) AS ctr_pctiles,
        approx_quantile(gsc_avg_position, [0.5, 0.75, 0.9, 0.95, 0.99]) AS position_pctiles,
        COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position != 0
      AND gsc_impressions > 0
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬─────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┐
│   impressions_pctiles   │                                 ctr_pctiles                                 │                                         position_pctiles                                          │    n    │
│         int64[]         │                                  double[]                                   │                                             double[]                                              │  int64  │
├─────────────────────────┼─────────────────────────────────────────────────────────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┤
│ [18, 66, 194, 347, 968] │ [0.0, 0.0, 0.0035622384784889573, 0.011649346052346325, 0.0503341250700415] │ [7.958087436023552, 21.3488448

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
con.sql(f"""
    WITH base AS (
        SELECT
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE
                WHEN gsc_avg_position BETWEEN 1 AND 10 THEN 'page_one'
                WHEN gsc_avg_position BETWEEN 11 AND 20 THEN 'headroom'
                ELSE 'rest'
            END AS position_tier,
            CASE WHEN gsc_impressions >= 194 THEN 'high_impressions' ELSE 'typical_impressions' END AS impression_bucket
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
          AND gsc_avg_position != 0
    )
    SELECT
        position_tier,
        impression_bucket,
        COUNT(*) AS n,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 3) AS weighted_ctr_pct,
        CASE WHEN COUNT(*) < 50 THEN 'TOO SMALL (<50)' ELSE 'ok' END AS sample_flag
    FROM base
    GROUP BY position_tier, impression_bucket
    ORDER BY position_tier, impression_bucket
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬─────────────────────┬─────────┬───────────────────┬──────────────┬──────────────────┬─────────────┐
│ position_tier │  impression_bucket  │    n    │ total_impressions │ total_clicks │ weighted_ctr_pct │ sample_flag │
│    varchar    │       varchar       │  int64  │      int128       │    int128    │      double      │   varchar   │
├───────────────┼─────────────────────┼─────────┼───────────────────┼──────────────┼──────────────────┼─────────────┤
│ headroom      │ high_impressions    │   27029 │          11527129 │        39950 │            0.347 │ ok          │
│ headroom      │ typical_impressions │  425219 │          13738224 │        38703 │            0.282 │ ok          │
│ page_one      │ high_impressions    │  232831 │         122835718 │       419431 │            0.341 │ ok          │
│ page_one      │ typical_impressions │ 1685916 │          61942240 │       219056 │            0.354 │ ok          │
│ rest          │ high_impressions    │   84840 │       

In [7]:
con.sql(f"""
    WITH base AS (
        SELECT
            gsc_impressions,
            gsc_clicks,
            gsc_clicks / gsc_impressions AS row_ctr,
            CASE WHEN gsc_impressions >= 194 THEN 'high_impressions' ELSE 'typical_impressions' END AS impression_bucket
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
          AND gsc_avg_position BETWEEN 1 AND 10
    )
    SELECT
        impression_bucket,
        COUNT(*) AS n,
        SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) AS n_zero_click,
        ROUND(100.0 * SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_zero_click,
        SUM(CASE WHEN gsc_clicks = 0 THEN gsc_impressions ELSE 0 END) AS impressions_stuck_at_zero_click,
        approx_quantile(row_ctr, [0.25, 0.5, 0.75, 0.9]) AS row_ctr_pctiles
    FROM base
    GROUP BY impression_bucket
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────┬─────────┬──────────────┬────────────────┬─────────────────────────────────┬──────────────────────────────────────────────────────────────────────────┐
│  impression_bucket  │    n    │ n_zero_click │ pct_zero_click │ impressions_stuck_at_zero_click │                             row_ctr_pctiles                              │
│       varchar       │  int64  │    int128    │     double     │             int128              │                                 double[]                                 │
├─────────────────────┼─────────┼──────────────┼────────────────┼─────────────────────────────────┼──────────────────────────────────────────────────────────────────────────┤
│ typical_impressions │ 1685916 │      1518843 │          90.09 │                        47718384 │ [0.0, 0.0, 0.0, 0.002416527258829508]                                    │
│ high_impressions    │  232831 │        95017 │          40.81 │                        36936460 │ [0.0, 0.00216178760807708

In [8]:
IMPRESSION_THRESHOLD = 194  # P90 of gsc_impressions, gsc_data_available IS TRUE, March 2026

preview = con.sql(f"""
    SELECT
        content_hash_id,
        report_date,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
        gsc_impressions AS score,
        'zero_click_high_visibility' AS reason_code,
        'review_ctr' AS action_label
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position BETWEEN 1 AND 10
      AND gsc_impressions >= {IMPRESSION_THRESHOLD}
      AND gsc_clicks = 0
    ORDER BY score DESC
    LIMIT 10
""")
preview

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬─────────────┬────────────────────┬─────────────────┬────────────┬───────┬────────────────────────────┬──────────────┐
│     content_hash_id      │ report_date │  gsc_avg_position  │ gsc_impressions │ gsc_clicks │ score │        reason_code         │ action_label │
│         varchar          │    date     │       double       │      int64      │   int64    │ int64 │          varchar           │   varchar    │
├──────────────────────────┼─────────────┼────────────────────┼─────────────────┼────────────┼───────┼────────────────────────────┼──────────────┤
│ content_945d6ff91386c817 │ 2026-03-04  │  8.613947762791694 │           37368 │          0 │ 37368 │ zero_click_high_visibility │ review_ctr   │
│ content_34a70fea29d15f24 │ 2026-03-22  │  3.129405326523167 │           27410 │          0 │ 27410 │ zero_click_high_visibility │ review_ctr   │
│ content_757b1fa67827358d │ 2026-03-13  │  2.261644474379566 │           19301 │          0 │ 19301 │ zero_click_high

In [9]:
import os

IMPRESSION_THRESHOLD = 194  # P90 of gsc_impressions, gsc_data_available IS TRUE, March 2026

queue_df = con.sql(f"""
    SELECT
        content_hash_id,
        report_date,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
        gsc_impressions AS score,
        'zero_click_high_visibility' AS reason_code,
        'review_ctr' AS action_label
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position BETWEEN 1 AND 10
      AND gsc_impressions >= {IMPRESSION_THRESHOLD}
      AND gsc_clicks = 0
    ORDER BY score DESC
""").df()

print(f"Queue rows: {len(queue_df):,}")
print(f"Distinct content items: {queue_df['content_hash_id'].nunique():,}")
print(f"Score range: {queue_df['score'].min()} to {queue_df['score'].max()}")

os.makedirs("work/outputs", exist_ok=True)
queue_df.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue rows: 95,017
Distinct content items: 16,578
Score range: 194 to 37368
Wrote work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
top20 = queue_df.head(20).reset_index(drop=True)
print(f"Top 20 rows represent {top20['content_hash_id'].nunique()} distinct content items")
top20

Top 20 rows represent 15 distinct content items


,content_hash_id,report_date,gsc_avg_position,gsc_impressions,gsc_clicks,score,reason_code,action_label
0,content_945d6ff91386c817,2026-03-04,8.613948,37368,0,37368,zero_click_high_visibility,review_ctr
1,content_34a70fea29d15f24,2026-03-22,3.129405,27410,0,27410,zero_click_high_visibility,review_ctr
2,content_757b1fa67827358d,2026-03-13,2.261644,19301,0,19301,zero_click_high_visibility,review_ctr
3,content_0c5606abaaab3178,2026-03-04,4.112533,13827,0,13827,zero_click_high_visibility,review_ctr
4,content_046fc480045b88f5,2026-03-29,6.915650,13764,0,13764,zero_click_high_visibility,review_ctr
5,content_69379902126ff53f,2026-03-05,2.532566,13726,0,13726,zero_click_high_visibility,review_ctr
6,content_bf078007df823490,2026-03-29,1.039915,13253,0,13253,zero_click_high_visibility,review_ctr
7,content_bf078007df823490,2026-03-27,1.063922,11952,0,11952,zero_click_high_visibility,review_ctr
8,content_0bca6d9a85a9b408,2026-03-04,7.485694,11464,0,11464,zero_click_high_visibility,review_ctr
9,content_bf078007df823490,2026-03-26,1.328322,11440,0,11440,zero_click_high_visibility,review_ctr


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.